# Цифровые технологии в профессиональной деятельности
## Раздел 1. Текст-майнинг
## Семинар 1. Количественные методы и контент-анализ

В рамках данного семинара мы рассмотрим базовые подходы к количественному анализу текстов. 
Материалом послужит текст первой главы *Алисы в Стране чудес* в переводе Бориса Заходера.

**Цель занятия:** сформировать исследовательский пайплайн для автоматизированной обработки текста, начиная от загрузки сырых файлов и заканчивая построением частотных профилей и их визуализацией. На Алисе мы потренируемся токенизировать и лемматизировать текст, а на Посланиях поучимся сравнивать тексты по разным показателям.

### Предобработка текста

#### 0. Загрузка файла

In [ ]:
import requests
import regex as re
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import pymorphy3
from razdel import tokenize
from collections import Counter
from nltk.corpus import stopwords
from wordcloud import WordCloud


# nltk.download('stopwords', quiet=True) # раскомментировать при первом запуске

In [ ]:
url = r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/alice_in_wonderland_ch1.txt"

response = requests.get(url)

alice_text = response.text

print(len(alice_text))
print(alice_text[:323])

Посимвольный объём текста сообщает нам некоторую информацию о его величине, но в действительности среди всех символов этого файла значительная часть пробельные отступы. Привычнее и понятнее получать какую-то количественную информацию о тексте на основе подсчёта слов. Попробуем посчитать слова с помощью встроенных функций Python.

#### 1. Предобработка текста: токенизация по словам

Для того чтобы проанализировать содержание, текст нужно разбить на отдельные слова — **токены**.

In [ ]:
naive_tokens = alice_text.split()

print("Как выглядят наши токены:")
print(naive_tokens[40:50])

Такие токены встречаются сплошь и рядом, а анализируя философский текст или исторический источник, а тем более художественный или искусствоведческий текст проблем токенизации может оказаться сильно больше. 

К тому же сам алгоритм очень жадный и ресурсоёмкий. Для обработки 2 тысяч токенов сойдёт, но когда в тексте или в целом корпусе миллионы токенов, процесс сильно замедлится.

Чтобы избежать этого, используем специализированные библиотеки для NLP (Natural Language Processing), обработки естественного языка.

Сегодня мы будем использовать библиотеки:
1) `razdel` (для токенизации) 

Хотя бы полистайте документацию: [Ссылка](https://github.com/natasha/razdel)

2) `pymorphy3` (для морфологического анализа).
3) `nltk` (возьмём из него стоп-слова)

Однако существует их больше! Отличаются они по качеству работы с разными языками, функциям. Другие известные библиотеки для токенизации и лемматизации попробуем захватить на следующих семинарах!

In [ ]:
# !pip install razdel pymorphy3

In [ ]:
# from razdel import tokenize

tokens = [substring.text for substring in tokenize(alice_text)]

print(f"Naive: {naive_tokens[15:25]}")
print(f"Smart: {tokens[16:30]}")

In [ ]:
substring = [substring for substring in tokenize(alice_text)][0]

print("\nЧто такое substring:")
print(type(substring), "— особый класс из библиотеки razdel")

In [ ]:
substring.start # обозначает начальный индекс в строке

In [ ]:
substring.stop # обозначает конечный индекс в строке (не включительно)

In [ ]:
substring.text # содержит текст между начальным и конечным индексами

Получив список таких объектов мы легко разберём их и получим список токенов с помощью list comprehensions.

In [ ]:
# from collections import Counter

tokens = [substring.text.lower() for substring in tokenize(alice_text)] # if substring.text.isalnum()]
word_counts = Counter(tokens)

print("\nКак теперь выглядят токены:")
print(tokens[:10])

print("\nВсего токенов:")
print(len(tokens))
print("\nСамые частотные токены:")
print(word_counts.most_common(10))

In [ ]:
# без функции .isalnum()

print(tokens[197:213])

In [ ]:
# с функцией .isalnum()

print(tokens[148:156])

Как мы увидели, простые методы либо удаляют дефисы вместе с пунктуацией, либо оставляют лишние знаки препинания. 

Для качественного анализа нам нужно использовать **регулярные выражения (с помощью продвинутой библиотеки RegEx)**, чтобы различать «хорошие» дефисы (в словах *кто-то*, *социально-экономический*) и «плохие» (повторы типа *да-да*).

Тренажёр для изучения и закрепления регулярных выражений: https://regex101.com/

#### 2. Разбор регулярных выражений

Есть старая шутка, ее приписывают программисту Джейми Завински: если у вас есть проблема, *и вы собираетесь ее решать при помощи регулярных выражений, то у вас две проблемы. Регулярные выражения – это формальный язык, который используется для того, чтобы находить, извлекать и заменять части текста*.

In [ ]:
# import regex as re

**ЧАЩЕ ВСЕГО РАБОТАТЬ ВЫ БУДЕТЕ С ДВУМЯ МЕТОДАМИ:**
- `re.findall()` — ищет в тексте все фрагменты, которые подходят под описание, и возвращает их в виде списка

- `re.sub()` — ищет паттерн и заменяет его на другой (в том числе на пустую строку)

**ХОРОШО ЗНАТЬ И ДРУГИЕ:**
- `re.search()` — ищет первое совпадение в строке по заданному паттерну
- `re.match()` — ищет паттерн только по началу строки
- `re.fullmatch()` — ищет совпадение со всем текстом
- `re.finditer()` — возвращает все совпадения в виде итератора

**Регулярное выражение** — это строка, паттерн для поиска, но со спецсимволами.

In [ ]:
example = "Алиса-то не так уж удивилась, даже когда 5 раз услыхала, что Кролик сказал (а сказал он 'ай-ай-ай! уже 11:10! Я опаздываю!').\n"

In [ ]:
# СПЕЦЗНАКИ С БЭКСЛЕШЕМ
r"\w" # любая буква (или цифра)
r"\d" # любая цифра
r"\s" # любой пробельный символ (пробел, табуляция, перенос строки)

r"\W" # любая не буква
r"\D" # любая не цифра
r"\S" # любой непробельный символ (пробел, табуляция, перенос строки)

r'.' # любой символ

re.findall(r'\d', example)

In [ ]:
# ТОЧНОЕ КОЛИЧЕСТВО
"а{3,5}" # от 3 до 5 раз повторяется "а"

In [ ]:
# 1. re.findall() — найти все совпадения. 
# Поищем все группы цифр (например, годы или время)
# \d — любая цифра, + — одна или более
numbers = re.findall(r'\d+', example)
print(f"Цифры в тексте: {numbers}")

# 2. Поиск имен собственных
# [А-Я] — любая заглавная русская буква, [а-я]+ — одна или более строчных
names = re.findall(r'[А-Я][а-я]+', example)
print(f"Имена: {names}")

# 3. Шаблон: цифры-двоеточие-цифры
time_pattern = re.findall(r'\d{1,2}:\d{2}', example)
print(f"Время: {time_pattern}")

In [ ]:
# 2. re.sub() — заменить все совпадения. 
# Заменим все группы цифр что-то одно
res = re.sub(r'\d+', r'505', example)
print(res)

In [ ]:
# В АВТОЗАМЕНЕ СКОБКИ ОЗНАЧАЮТ ГРУППЫ

"\0" # всё выражение
"\1" # первая группа

re.sub(r"(\w+) (\w+)", r"\2 \1", "мама мыла")

Подробный разбор регулярных выражений вам нужно будет осуществить дома, это потребуется для выполнения домашнего задания. К сожалению, эта тема слишком обширна, чтобы задерживаться на ней в рамках нашего курса. Вопросы по этой теме можно будет задавать в личные сообщения преподавателю или в чате курса!

In [ ]:
# вернёмся к нашей Алисе
# вариант 1
double_tokens = []

for token in tokens:
    if re.search(r'(\w+)-(\1)', token):
        double_tokens.append(token)

print(double_tokens)



In [ ]:
# решение

double_tokens = []

clean_tokens = []

for token in tokens:
    match = re.search(r'(\w+)-(\w+)', token)
    if match:
        if match.group(1) == match.group(2): 
            double_tokens.append(token)
            clean_tokens.extend(token.split('-'))
    else:
        clean_tokens.append(token)

print("Редуплицированные токены:", double_tokens)
print("Итог:", clean_tokens[196:213])

Итак, у нас появился список токенов, мы можем посчитать их вплоть до знаков препинания. При этом не все содержательные количественные показатели будут точными. Например, если мы захотим посчитать все упоминания имени Алиса, нам нужно будет прописать и посчитать все словоформы: Алиса, Алисе, Алисой... и так для каждого интересующего нас слова.

Вместо этого увлекательного занятия мы можем воспользоваться автоматической лемматизацией: приведением всех токенов текста к начальной форме.

#### 3. Предобработка текста: лемматизация

Библиотека **pymorphy3** — это стандарт де-факто для автоматического морфологического анализа русского языка в среде Python. Её создатель, программист и лингвист-разработчик Михаил Коробов, спроектировал инструмент так, чтобы он был максимально быстрым и не требовал огромных вычислительных мощностей. 

В основе работы лежит словарь [OpenCorpora](https://opencorpora.org/dict.php), содержащий сотни тысяч лексем с их грамматическими формами. Когда мы передаем библиотеке слово, она не просто ищет его в списке, а сопоставляет с хранящимися в памяти морфемами. Благодаря этому pymorphy3 умеет восстанавливать начальную форму слов (лемму) и определять часть речи, число, род или падеж даже у тех слов, которых изначально не было в словаре, — программа анализирует их по аналогии с похожими по структуре известными словами.

In [ ]:
# Как работает лемматизация в pymorphy3
# import pymorphy3

morph = pymorphy3.MorphAnalyzer() # создаём морфологический парсер (parse — анализ, разбор)

word = 'бармоглота'  # выберем слово для разбора

lemma_options = morph.parse(word) # с помощью метода parse выведем все вероятностные разборы
lemma_options

In [ ]:
lemma_options[0].word # в разборе остаётся сама словоформа

In [ ]:
lemma_options[0].score # у каждого разбора есть вероятностная оценка

In [ ]:
lemma_options[0].tag # морфологические теги

In [ ]:
lemma_options[0].tag.POS

| Атрибут       | Граммема | Описание на русском               | Примеры значений                       |
|---------------|----------------|-----------------------------------|----------------------------------------|
| .POS          | Part of Speech | Часть речи (самый важный атрибут) | NOUN (сущ), VERB (глаг), ADJF (прилаг) |
| .animacy      | Animacy        | Одушевленность                    | anim (одуш), inan (неодуш)             |
| .gender       | Gender         | Род                               | masc (муж), fem (жен), neut (ср)       |
| .number       | Number         | Число                             | sing (ед), plur (мн)                   |
| .case         | Case           | Падеж                             | nomn (им), gent (род), datv (дат)      |
| .aspect       | Aspect         | Вид глагола                       | perf (сов), impf (несов)               |
| .transitivity | Transitivity   | Переходность глагола              | tran (перех), itran (неперех)          |
| .person       | Person         | Лицо                              | 1per, 2per, 3per                       |
| .tense        | Tense          | Время                             | pres (наст), past (прош), futr (буд)   |
| .mood         | Mood           | Наклонение                        | indc (изъяв), impr (повелит)           |
| .voice        | Voice          | Залог                             | actv (действ), pssv (страд)            |

Все теги-граммемы можно изучить на сайте: https://opencorpora.org/dict.php?act=gram

In [ ]:
lemma_options[0].normal_form # лемма, начальная форма слова

Итак, нетрудно составить цикл и прогнать через него все токены, получить готовый список лемм.

In [ ]:
lemmas = []

for token in clean_tokens:
    lemma = morph.parse(token)[0].normal_form
    lemmas.append(lemma)

lemma_count = Counter(lemmas)

print("В результате получили вот такой список лемм:")
print(lemmas[:9])
print("\nВсего слов в тексте:")
print(len(lemmas))
print("\nВсего уникальных слов в тексте:")
print(len(set(lemmas)))
print("\nСамые частотные слова в тексте:")
print(lemma_count.most_common(10))

Среди самых частотных слов в тексте в основном служебные слова, местоимения. По этой причине теряются значимые понятия, ключевые слова текста. Попробуем очистить список лемм от слов, не несущих большой смысловой нагрузки их принято называть стоп-словами. У каждого языка их список отличается, в каждом тексте их список можно увеличивать по необходимости. Мы же воспользуемся готовым списком из `nltk`, известной библиотекой для задач NLP.

In [ ]:
# from nltk.corpus import stopwords
# import nltk

# Загружаем список стоп-слов
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('russian'))

print(list(stop_words)[:10])
print(len(stop_words))

In [ ]:
# пройдёмся циклом по леммам

cleaned_lemmas = []

for lemma in lemmas:
    if lemma not in stop_words:
        cleaned_lemmas.append(lemma)

lemma_count = Counter(cleaned_lemmas)

print("Всего слов в тексте:")
print(len(cleaned_lemmas))
print("\nВсего уникальных слов в тексте:")
print(len(set(cleaned_lemmas)))
print("\nСамые частотные слова в тексте:")
lemma_count.most_common(20)

In [ ]:
stop_words.update(['который', 'это', 'наш', 'свой', 'быть', 'мочь', 'весь', 'всё', 'год', 'который', 'ещё'])

cleaned_lemmas = []

for lemma in lemmas:
    if lemma not in stop_words:
        cleaned_lemmas.append(lemma)

lemma_count = Counter(cleaned_lemmas)

print("Всего слов в тексте:")
print(len(cleaned_lemmas))
print("\nВсего уникальных слов в тексте:")
print(len(set(cleaned_lemmas)))
print("\nСамые частотные слова в тексте:")
lemma_count.most_common(20)

Главное преимущество этой библиотеки — высокая скорость и умение догадываться о форме незнакомых слов по их окончаниям, что крайне полезно при работе с неологизмами или, напротив, с незафиксированными в словаре архаизмами. 
Она очень легкая, не требует видеокарты и предоставляет исчерпывающую информацию о грамматических категориях (род, число, падеж).

Однако основным недостатком является отсутствие учета контекста: библиотека анализирует каждое слово в отдельности, поэтому не может отличить существительное «стекло» от глагола «стекло». 
Кроме того, словари OpenCorpora, на которых она основана, обновляются редко, из-за чего некоторые специфические термины могут лемматизироваться с ошибками.

In [ ]:
lemma = morph.parse('Алисин')
[option.normal_form for option in lemma] # наиболее вероятный вариант - ошибочный :(

### Формирование корпуса текстов

Для начала нам необходимо загрузить текстовые файлы в оперативную память. 
Наиболее удобной структурой данных для хранения корпуса в Python является словарь (`dict`), где ключом будет выступать идентификатор текста (например, год), а значением — сам текст в виде строки.

In [ ]:
# Список файлов, с которыми мы будем работать
urls = [
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_1994.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_1998.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2002.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2006.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2010.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2014.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2018.txt",
    r"https://raw.githubusercontent.com/always1ready/dh-tools/refs/heads/main/data/01_seminar/federal_assembly_address_2024.txt",
]

corpus = {}

for url in urls:
    text = requests.get(url).text
    year = re.search('\d{4}', url).group()
    corpus[year] = text

print("Послания за гг.:", list(corpus.keys()))

Посмотрим на базовую статистику: как менялся объём текстов посланий с течением времени. 

Воспользуемся функцией `len()`, которая для строкового типа данных возвращает количество символов.

In [ ]:
for year, text in corpus.items():
    print(f"Год {year} -> {len(text)} символов")

longest_text = max([(year, len(text)) for year, text in corpus.items()])
print(f"\nСамое длинное послание было в {longest_text[1]} году ({longest_text[0]} символов).")

                                                                        # Обратите внимание на два получившихся кортежа. 
                                                                        # Объясните, почему функция выдаёт разные результаты
                                                                        
longest_text = max([(len(text), year) for year, text in corpus.items()])
print(f"\nСамое длинное Послание было в {longest_text[1]} году ({longest_text[0]} символов).")

#### Задание для самостоятельной работы

Допишите функцию ниже, чтобы она провела токенизацию и лемматизацию нашего корпуса. Пишите код в `ВАШ КОД` ниже:

In [ ]:
def process_corpus(corpus_dict):
    full_results = {}
    
    for year, text in corpus_dict.items():
        # 1. Токенизация (используем razdel)
        #
        #
        # 2. Лемматизация и фильтрация стоп-слов
        #
        #
            # Получаем нормальную форму слова
            #
            # Проверяем: не стоп-слово ли это и состоит ли оно из более чем двух символов
            #
        # 3. Сохраняем результат как Counter для этого года
        #
        
    return full_results

# Запускаем обработку всего корпуса
final_counts = process_corpus(corpus)

final_counts

Давайте превратим наши данные в DataFrame, чтобы сравнить тексты Посланий разных лет статистически.

`Counter()` - это словарь. Чтобы превратить словарь в таблицу, мы используем его метод `.items()`, который превращает словарь в список пар вида «слово — число». 

Когда мы передаем словарь в функцию `pd.DataFrame()`, библиотека `pandas` просто берет каждую такую пару и кладет её в отдельную строчку: первое значение из пары (ключ) отправляется в левую колонку, а второе (значение) — в правую.

Обратите внимание на левый столбик с цифрами — индексами. `Pandas` добавляет их автоматически, это его отличительная особенность (которая иногда приносит много хлопот).

In [ ]:
import pandas as pd

In [ ]:
dict = {
    'Земля' : '12 742',
    'Марс' : '6 779',
    'Юпитер' : '139 820',
    'Сатурн' : '116 460',
}

# Мы передаем список пар (dict.items()) и сразу называем колонки.
pd.DataFrame(dict.items(), columns=['Планета', 'Диаметр (км)'])

Но у нас в переменной **словарь** словарей, причём с частично совпадающим набором ключей. 

Чему будут соответствовать элементы словаря верхнего уровня и чему элементы вложенного словаря?

In [ ]:
df = pd.DataFrame(final_counts).fillna(0).astype(int)
df.head(10)

### Документация к используемым библиотекам:
* razdel: https://github.com/natasha/razdel
* pymorphy3: https://pymorphy3.readthedocs.io/
* nltk: https://www.nltk.org/
* regex: https://pypi.org/project/regex/ 
* pandas: https://pandas.pydata.org/docs/
* matplotlib: https://matplotlib.org/stable/contents.html

### Полезное
* Интерактивный тренажер регулярных выражений: https://regex101.com/
* Система граммем словаря OpenCorpora: https://opencorpora.org/dict.php?act=gram
* Фридл Д. - Регулярные выражения